# RTE on Kaggle T4 x2: ALBERT-large fine-tuning (FreeLB / SMART / distillation / self-distillation / seed sweeps)

**Settings:** Accelerator **GPU T4 x2**, Internet **on**. Add your dataset with `train.jsonl` and `val.jsonl`
(and `test.jsonl` when you get it) as an input.

**How a session works**
1. Set `PHASES` in the config cell (below) to the phase(s) for this session.
2. **Save Version -> Save & Run All (Commit)**. It runs in the background (up to 12 h). Interactive sessions lose
   their files when they end; commits keep them as the version's output.
3. Next session: **Add Input -> Notebook Output -> this notebook's latest version**, change `PHASES`, commit again.
   Earlier runs are copied back in and skipped. A phase that did not finish just continues where it stopped.

| Session | Config used | What ran | Time |
|---|---|---|---|
| 1 | `PHASES = ["check", "1"]` | ALBERT-large recipes (base, long, FreeLB, SMART) + DeBERTa-v3-large teacher runs, 5 seeds each | ~6.2 h |
| 2 | `PHASES = ["2", "3"]`, `SELF_FROM = "freelb"` | distillation from DeBERTa-large (kd, kd_freelb), then self-distillation (self, self_freelb) | ~5 h |
| 3 | `PHASES = ["6"]` (CPU is enough) | soups, summary tables, dev predictions + error slices | ~5 min |

Phases 4 and 5 (more seeds, size ladder) exist in this notebook but were not run for the report.
Times are estimates for 5 seeds; the `min` column of the summary shows the real minutes per run after session 1.
Before sessions 2-3, read the summary table and set `R`, `R2`, `SELF_FROM` in the config cell.
New runs stop being started after `SESSION_HOURS`, so a commit never hits Kaggle's 12-hour limit.

In [ ]:
import os
CODE_DIR = os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working") + "/rte_scripts"
os.makedirs(CODE_DIR, exist_ok=True)

## 1. Scripts (written to /kaggle/working/rte_scripts - no need to edit)

In [ ]:
%%writefile {CODE_DIR}/common.py
"""Shared code for the RTE experiments: data, models, the training loop (plain / FreeLB / SMART / distillation),
evaluation and file layout. Every other script imports this module."""
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")       # deterministic cuBLAS (set before CUDA starts)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")      # keep run logs readable
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import contextlib
import csv
import glob
import json
import math
import random
import re
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

LABEL2ID = {"entailment": 0, "not_entailment": 1}
ID2LABEL = {0: "entailment", 1: "not_entailment"}
MODEL_KEYS = ("input_ids", "attention_mask", "token_type_ids")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAILED_CE = 0.5          # a run FAILED if the gold-label training loss at its saved checkpoint is still >= this

# ---------------------------------------------------------------- paths
DATA_DIR = os.environ.get("RTE_DATA_DIR", "data")
SMOKE = False            # --smoke: tiny data, 1 epoch, separate output folder (pipeline check only)


def enable_smoke():
    global SMOKE
    SMOKE = True


def work_dir():
    return os.environ.get("RTE_WORK_DIR", "work") + ("_smoke" if SMOKE else "")


def runs_dir(): return os.path.join(work_dir(), "runs")
def teachers_dir(): return os.path.join(work_dir(), "teachers")
def soups_dir(): return os.path.join(work_dir(), "soups")
def pruned_dir(): return os.path.join(work_dir(), "pruned")
def summary_dir(): return os.path.join(work_dir(), "summary")
def preds_dir(): return os.path.join(work_dir(), "predictions")
def run_dir(exp, seed): return os.path.join(runs_dir(), exp, f"s{seed}")


def resolve_model(hub_name):
    """Hub name -> what from_pretrained should load: 'local:<dir>' (our continued-pretraining outputs) -> <dir>;
    otherwise the hub name, or a local path if RTE_MODEL_MAP points to a JSON map (offline machines / tests)."""
    if hub_name.startswith("local:"):
        return hub_name[len("local:"):]
    path = os.environ.get("RTE_MODEL_MAP")
    return json.load(open(path)).get(hub_name, hub_name) if path else hub_name


# ---------------------------------------------------------------- data
def parse_label(raw):
    if raw is None or raw == "" or str(raw) == "-1":
        return None
    if isinstance(raw, str) and raw.strip() in LABEL2ID:
        return LABEL2ID[raw.strip()]
    return int(raw)


def read_examples(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".tsv", ".csv"):
        with open(path, encoding="utf-8", newline="") as f:
            recs = list(csv.DictReader(f, delimiter="\t" if ext == ".tsv" else ",",
                                       quoting=csv.QUOTE_NONE if ext == ".tsv" else csv.QUOTE_MINIMAL))
    else:
        text = open(path, encoding="utf-8").read().strip()
        try:
            recs = json.loads(text)
            recs = recs if isinstance(recs, list) else recs.get("data", [])
        except json.JSONDecodeError:
            recs = [json.loads(line) for line in text.splitlines() if line.strip()]
    return [{"idx": r.get("idx", r.get("index", i)),
             "premise": str(r.get("premise", r.get("sentence1"))).strip(),
             "hypothesis": str(r.get("hypothesis", r.get("sentence2"))).strip(),
             "label": parse_label(r.get("label"))} for i, r in enumerate(recs)]


def find_data(kind):
    names = {"train": ["train"], "dev": ["val", "dev", "validation"], "test": ["test"]}[kind]
    for n in names:
        for ext in (".jsonl", ".json", ".tsv", ".csv"):
            p = os.path.join(DATA_DIR, n + ext)
            if os.path.exists(p):
                return p
    return None


def stratified_split(examples, ratio, seed):
    rng = random.Random(seed)
    train, hold = [], []
    for lab in (0, 1):
        group = [e for e in examples if e["label"] == lab]
        rng.shuffle(group)
        n = int(round(len(group) * ratio))
        hold += group[:n]
        train += group[n:]
    rng.shuffle(train)
    return train, hold


def load_split(cfg):
    """The 90/10 train / hold-out split of train.jsonl. The same split for every model, recipe and seed."""
    path = find_data("train")
    assert path, f"no train file in {DATA_DIR}/ (expected train.jsonl)"
    labelled = [e for e in read_examples(path) if e["label"] is not None]
    if SMOKE:
        labelled = labelled[:80]
    return stratified_split(labelled, cfg["val_ratio"], cfg["split_seed"])


def load_dev():
    path = find_data("dev")
    if not path:
        return None
    ex = read_examples(path)
    assert all(e["label"] is not None for e in ex), f"{path} has no labels"
    return ex[:24] if SMOKE else ex


def load_test():
    path = find_data("test")
    if not path:
        return None
    ex = read_examples(path)
    return ex[:8] if SMOKE else ex


def augment(examples, n, p, seed):
    """Rule-based perturbed copies for distillation (no language model involved): every word is deleted with
    probability p/2 or replaced by [MASK] with probability p/2. Copies carry NO gold label - a perturbation can
    change the answer - so the teacher's probabilities are their only target."""
    rng = random.Random(seed + 7)

    def perturb(text):
        out = []
        for w in text.split():
            r = rng.random()
            if r < p / 2:
                continue
            out.append("[MASK]" if r < p else w)
        return " ".join(out) if out else text

    return [{"idx": f"aug{k}_{e['idx']}", "premise": perturb(e["premise"]), "hypothesis": perturb(e["hypothesis"]),
             "label": None} for k in range(n) for e in examples]


class RTEDataset(Dataset):
    """[CLS] premise [SEP] hypothesis [SEP], truncated to max_len (the longer text is cut first)."""

    def __init__(self, examples, tokenizer, max_len):
        enc = tokenizer([e["premise"] for e in examples], [e["hypothesis"] for e in examples],
                        truncation=True, max_length=max_len)
        keys = [k for k in MODEL_KEYS if k in enc]
        self.items = []
        for i, e in enumerate(examples):
            item = {k: enc[k][i] for k in keys}
            item["labels"] = e["label"] if e["label"] is not None else -100
            if "teacher" in e:
                item["teacher"] = e["teacher"]
            self.items.append(item)

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def make_collate(tokenizer):
    def collate(batch):                                   # dynamic padding to the longest pair in the batch
        out = dict(tokenizer.pad([{k: b[k] for k in MODEL_KEYS if k in b} for b in batch],
                                 padding=True, return_tensors="pt"))
        out["labels"] = torch.tensor([b["labels"] for b in batch], dtype=torch.long)
        if "teacher" in batch[0]:
            out["teacher"] = torch.tensor([b["teacher"] for b in batch], dtype=torch.float)
        return out
    return collate


def loader(examples, tokenizer, cfg, shuffle=False, seed=0, batch_size=None):
    ds = RTEDataset(examples, tokenizer, cfg["max_len"])
    gen = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(ds, batch_size=batch_size or (cfg["batch_size"] if shuffle else 64), shuffle=shuffle,
                      generator=gen, collate_fn=make_collate(tokenizer))


# ---------------------------------------------------------------- teachers (distillation)
def teacher_path(name): return os.path.join(teachers_dir(), f"{name}.jsonl")


def load_teacher_examples(name, train_ex):
    """Training pairs + perturbed copies, each with the teacher's log-probabilities (written by teacher.py)."""
    path = teacher_path(name)
    if not os.path.exists(path):
        raise SystemExit(f"teacher '{name}' not found ({path}) - run teacher.py --name {name} ... first")
    rows = [json.loads(line) for line in open(path)]
    gold_idx = sorted(str(r["idx"]) for r in rows if r["label"] is not None)
    assert gold_idx == sorted(str(e["idx"]) for e in train_ex), \
        "teacher file was built on a different train/hold-out split - rebuild it with teacher.py"
    return [{"idx": r["idx"], "premise": r["premise"], "hypothesis": r["hypothesis"], "label": r["label"],
             "teacher": r["teacher_logp"]} for r in rows]


# ---------------------------------------------------------------- models
# Challenge rule: start only from PRETRAINED checkpoints, never from one already fine-tuned on RTE / NLI / instructions.
FORBIDDEN_NAME_PARTS = ("mnli", "snli", "anli", "xnli", "nli", "rte", "glue", "superglue", "flan", "instruct")


def check_rules(name, loading_info):
    low = name.lower().split("/")[-1]
    bad = [p for p in FORBIDDEN_NAME_PARTS if p in low]
    assert not bad, f"RULES: '{name}' looks NLI/RTE/instruction fine-tuned ({bad}) - not allowed"
    missing = [str(k) for k in loading_info.get("missing_keys", [])]
    assert any("classifier" in k for k in missing), \
        f"RULES: '{name}' already contains a classification head (fine-tuned on some task) - not allowed"
    return sorted(k for k in missing if "classifier" in k or "pooler" in k)


def build_model(cfg):
    model, info = AutoModelForSequenceClassification.from_pretrained(
        resolve_model(cfg["hub"]), num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID,
        output_loading_info=True, **cfg.get("model_kwargs", {}))
    new_head = check_rules(cfg["hub"], info)
    print(f"rules check passed: {cfg['hub']} is pretrained-only; newly initialised head: {new_head}")
    return model.float()                                 # fp32 master weights (some checkpoints are stored in fp16)


def count_params(model): return sum(p.numel() for p in model.parameters())   # shared ALBERT weights counted once


def load_model_dir(path):
    model = AutoModelForSequenceClassification.from_pretrained(path).float().to(DEVICE).eval()
    return model, AutoTokenizer.from_pretrained(path)


def amp_dtype(cfg=None):
    """bf16 on GPUs that have it (A100, H100/H200, GB10); fp16 + loss scaling on older ones (T4, P100, V100)."""
    if DEVICE.type != "cuda" or (cfg is not None and not cfg.get("use_amp", True)):
        return None
    # NOT torch.cuda.is_bf16_supported(): it also returns True when bf16 is only EMULATED (T4), which is very slow
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def gpu_memory_gb():
    return torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE.type == "cuda" else float("inf")


def make_scaler(dtype):
    if dtype != torch.float16:
        return None
    try:
        return torch.amp.GradScaler("cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler()


def backward(x, scaler):
    (scaler.scale(x) if scaler is not None else x).backward()


def finite(t):
    return torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)


def autocast(dtype):
    return torch.autocast("cuda", dtype=dtype) if dtype is not None else contextlib.nullcontext()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


@torch.no_grad()
def predict(model, dl, dtype=None):
    """Returns (logits list, mean CE loss or None, accuracy or None)."""
    model.eval()
    logits_all, tot, correct, n = [], 0.0, 0, 0
    for batch in dl:
        inputs = {k: batch[k].to(DEVICE) for k in MODEL_KEYS if k in batch}
        with autocast(dtype):
            logits = model(**inputs).logits.float()
        logits_all += logits.cpu().tolist()
        labels = batch["labels"].to(DEVICE)
        has = labels != -100
        if has.any():
            tot += F.cross_entropy(logits[has], labels[has], reduction="sum").item()
            correct += (logits[has].argmax(-1) == labels[has]).sum().item()
            n += has.sum().item()
    return logits_all, (tot / n if n else None), (correct / n if n else None)


def accuracy(logits, examples):
    return sum(int(np.argmax(l)) == e["label"] for l, e in zip(logits, examples)) / len(examples)


# ---------------------------------------------------------------- losses
def task_loss(logits, labels, teacher, cfg):
    """Gold pairs: cross-entropy (or alpha*KD + (1-alpha)*CE with a teacher). Perturbed copies: KD only.
    Returns (loss, summed gold CE, number of gold pairs)."""
    has = labels != -100
    ce_each = F.cross_entropy(logits, labels.clamp(min=0), reduction="none", label_smoothing=cfg["label_smoothing"])
    ce_each = ce_each * has
    if teacher is None:
        loss = ce_each.sum() / has.sum().clamp(min=1)
    else:
        T, a = cfg["kd_T"], cfg["kd_alpha"]
        t = F.log_softmax(teacher / T, -1)
        kd_each = (t.exp() * (t - F.log_softmax(logits / T, -1))).sum(-1) * T * T      # KL(teacher || student)
        loss = torch.where(has, a * kd_each + (1 - a) * ce_each, kd_each).mean()
    return loss, ce_each.detach().sum().item(), int(has.sum().item())


def sym_kl(p_logits, q_logits):
    p, q = F.log_softmax(p_logits, -1), F.log_softmax(q_logits, -1)
    return ((p.exp() * (p - q)).sum(-1) + (q.exp() * (q - p)).sum(-1)).mean()


def unit(x):  # per-token unit vectors
    return x / (x.norm(dim=-1, keepdim=True) + 1e-12)


def train_step(model, batch, cfg, accum, dtype, scaler=None):
    """Forward + backward for one batch (gradients are accumulated, the optimizer step happens outside).
    adv="none":   plain training.
    adv="freelb": FreeLB (Zhu et al., ICLR 2020): K ascent steps on a perturbation of the word embeddings; the
                  parameter gradients of all K perturbed passes are accumulated ("free" adversarial training).
    adv="smart":  SMART-style smoothness term (Jiang et al., ACL 2020): CE on the clean input + alpha * symmetric KL
                  between clean and adversarially perturbed predictions (one ascent step).
    Perturbation sizes are RELATIVE to each token's embedding norm, so the same values work for ALBERT and DeBERTa."""
    inputs = {k: batch[k].to(DEVICE) for k in MODEL_KEYS if k in batch}
    labels = batch["labels"].to(DEVICE)
    teacher = batch["teacher"].to(DEVICE) if "teacher" in batch else None
    adv = cfg["adv"]
    if adv == "none":
        with autocast(dtype):
            logits = model(**inputs).logits.float()
        loss, ce_sum, n_gold = task_loss(logits, labels, teacher, cfg)
        backward(loss / accum, scaler)
        return loss.item(), ce_sum, n_gold

    emb_layer = model.get_input_embeddings()
    ids = inputs["input_ids"]
    rest = {k: v for k, v in inputs.items() if k != "input_ids"}
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    with torch.no_grad():
        norms = emb_layer(ids).norm(dim=-1, keepdim=True)            # per-token embedding norm

    if adv == "freelb":
        K = cfg["adv_steps"]
        delta = unit(torch.randn(*ids.shape, emb_layer.embedding_dim, device=DEVICE)) * cfg["adv_init"] * norms * mask
        total, first = 0.0, None
        for k in range(K):
            delta.requires_grad_()
            with autocast(dtype):
                logits = model(inputs_embeds=emb_layer(ids) + delta, **rest).logits.float()
            loss, ce_sum, n_gold = task_loss(logits, labels, teacher, cfg)
            backward(loss / (K * accum), scaler)
            total += loss.item() / K
            first = first or (ce_sum, n_gold)
            if k == K - 1:
                break
            delta = (delta.detach() + cfg["adv_lr"] * norms * unit(finite(delta.grad.detach()))) * mask
            if cfg["adv_max"] > 0:                                    # project onto the per-token ball
                scale = (cfg["adv_max"] * norms / (delta.norm(dim=-1, keepdim=True) + 1e-12)).clamp(max=1.0)
                delta = delta * scale
        return total, first[0], first[1]

    if adv == "smart":
        with autocast(dtype):
            logits = model(**inputs).logits.float()
        loss, ce_sum, n_gold = task_loss(logits, labels, teacher, cfg)
        emb = emb_layer(ids)
        noise = (unit(torch.randn_like(emb)) * 1e-3 * norms * mask).requires_grad_()
        with autocast(dtype):
            adv_logits = model(inputs_embeds=emb.detach() + noise, **rest).logits.float()
        g, = torch.autograd.grad(sym_kl(adv_logits, logits.detach()) * 1024.0, noise)   # scaled: fp16-safe, direction only
        noise = noise.detach() + cfg["adv_lr"] * norms * unit(finite(g))
        noise = noise * (cfg["adv_eps"] * norms / (noise.norm(dim=-1, keepdim=True) + 1e-12)).clamp(max=1.0) * mask
        with autocast(dtype):
            adv_logits = model(inputs_embeds=emb + noise, **rest).logits.float()
        total = loss + cfg["smart_alpha"] * sym_kl(adv_logits, logits)
        backward(total / accum, scaler)
        return total.item(), ce_sum, n_gold

    raise ValueError(f"unknown adv mode {adv}")


# ---------------------------------------------------------------- one training run
def eval_points(steps_per_epoch, epochs, per_epoch):
    pts = set()
    for e in range(epochs):
        for j in range(1, per_epoch + 1):
            pts.add(max(1, int(round(steps_per_epoch * (e + j / per_epoch)))))
    return pts


def train_run(cfg, seed):
    exp = cfg["experiment"]
    out = run_dir(exp, seed)
    if os.path.exists(os.path.join(out, "results.json")):
        print(f"skip {exp} seed {seed}: already trained")
        return json.load(open(os.path.join(out, "results.json")))
    os.makedirs(out, exist_ok=True)
    set_seed(seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    t_start = time.time()
    dtype = amp_dtype(cfg)
    scaler = make_scaler(dtype)

    tok = AutoTokenizer.from_pretrained(resolve_model(cfg["hub"]))
    train_ex, hold_ex = load_split(cfg)
    n_gold = len(train_ex)
    if cfg.get("teacher"):
        train_ex = load_teacher_examples(cfg["teacher"], train_ex)
    dev_ex, test_ex = load_dev(), load_test()

    train_dl = loader(train_ex, tok, cfg, shuffle=True, seed=seed)
    hold_dl = loader(hold_ex, tok, cfg)
    model = build_model(cfg).to(DEVICE)
    n_params = count_params(model)
    accum = cfg["grad_accum"]
    precision = str(dtype).split(".")[-1] if dtype is not None else "fp32"
    batch_txt = f"batch {cfg['batch_size']}" + (f" x {accum} accumulated" if accum > 1 else "")
    print(f"=== {exp} | seed {seed} | {cfg['hub']} | {n_params:,} params | train {n_gold} gold"
          f"{f' + {len(train_ex) - n_gold} teacher-labelled copies' if cfg.get('teacher') else ''} | hold-out {len(hold_ex)}"
          f" | adv {cfg['adv']} | lr {cfg['lr']} | {cfg['epochs']} epochs | {batch_txt} | {DEVICE} {precision} ===", flush=True)

    params = [p for p in model.parameters() if p.requires_grad]
    decay = [p for p in params if p.ndim >= 2]
    no_decay = [p for p in params if p.ndim < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": cfg["weight_decay"]},
                             {"params": no_decay, "weight_decay": 0.0}], lr=cfg["lr"], eps=1e-6)  # bias-corrected Adam
    steps_per_epoch = math.ceil(len(train_dl) / accum)
    total_steps = steps_per_epoch * cfg["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup_ratio"] * total_steps), total_steps)
    evals = eval_points(steps_per_epoch, cfg["epochs"], cfg["evals_per_epoch"])

    log, best, best_state = [], None, None
    step, w_loss, w_cnt, w_ce, w_gold, t0, diverged = 0, 0.0, 0, 0.0, 0, time.time(), False
    for epoch in range(cfg["epochs"]):
        model.train()
        opt.zero_grad(set_to_none=True)
        for i, batch in enumerate(train_dl):
            loss, ce_sum, ng = train_step(model, batch, cfg, accum, dtype, scaler)
            if not math.isfinite(loss):
                diverged = True
                break
            w_loss += loss
            w_cnt += 1
            w_ce += ce_sum
            w_gold += ng
            if (i + 1) % accum and i + 1 != len(train_dl):
                continue
            if scaler is not None:
                scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["max_grad_norm"])
            if scaler is not None:
                scaler.step(opt)                  # skipped automatically if fp16 gradients overflowed
                scaler.update()
            else:
                opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step not in evals:
                continue
            _, h_loss, h_acc = predict(model, hold_dl, dtype)
            row = {"step": step, "epoch": round(step / steps_per_epoch, 3), "train_loss": w_loss / max(w_cnt, 1),
                   "train_ce": w_ce / max(w_gold, 1), "holdout_loss": h_loss, "holdout_acc": h_acc}
            log.append(row)
            better = best is None or (h_acc, -h_loss) > (best["holdout_acc"], -best["holdout_loss"])
            if better:
                best = row
                best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
            print(f"step {step:5d} (epoch {row['epoch']:5.2f}) | train loss {row['train_loss']:.4f} gold CE "
                  f"{row['train_ce']:.4f} | hold-out loss {h_loss:.4f} acc {h_acc:.4f} | {time.time() - t0:.0f}s"
                  + ("  <- best" if better else ""), flush=True)
            w_loss, w_cnt, w_ce, w_gold = 0.0, 0, 0.0, 0
            model.train()
        if diverged:
            print("!! loss became NaN/inf - stopping this run", flush=True)
            break

    if best is None:                                   # diverged before the first evaluation
        best = {"step": 0, "epoch": 0.0, "train_loss": float("nan"), "train_ce": float("nan"),
                "holdout_loss": float("nan"), "holdout_acc": 0.0}
        best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    best_path = os.path.join(out, "best")
    model.save_pretrained(best_path)
    tok.save_pretrained(best_path)
    json.dump({**cfg, "seed": seed}, open(os.path.join(best_path, "run_config.json"), "w"), indent=2)
    json.dump(log, open(os.path.join(out, "train_log.json"), "w"), indent=2)

    res = {"experiment": exp, "model": cfg["model"], "recipe": cfg["recipe"], "family": cfg["family"],
           "hub": cfg["hub"], "seed": seed, "params": n_params, "best_step": best["step"], "best_epoch": best["epoch"],
           "holdout_acc": best["holdout_acc"], "holdout_loss": best["holdout_loss"],
           "train_ce_at_best": best["train_ce"], "final_train_ce": log[-1]["train_ce"] if log else float("nan"),
           "teacher": cfg.get("teacher"), "minutes": (time.time() - t_start) / 60}
    res["failed"] = not (best["train_ce"] < FAILED_CE)     # NaN (diverged before any checkpoint) also counts as failed
    if dev_ex:
        res["dev_logits"], _, res["dev_acc"] = predict(model, loader(dev_ex, tok, cfg), dtype)
    if test_ex:
        res["test_logits"], _, _ = predict(model, loader(test_ex, tok, cfg), dtype)
    json.dump(res, open(os.path.join(out, "results.json"), "w"))  # written last = the run is complete
    print(f"done: saved checkpoint at epoch {best['epoch']} | hold-out {best['holdout_acc']:.4f}"
          + (f" | dev {res['dev_acc']:.4f}" if dev_ex else "") + (" | FAILED (did not fit the training data)" if res["failed"] else "")
          + f" | {res['minutes']:.1f} min", flush=True)
    return res


# ---------------------------------------------------------------- reading finished runs
def all_results():
    out = []
    for p in glob.glob(os.path.join(runs_dir(), "*", "s*", "results.json")):
        r = json.load(open(p))
        r["dir"] = os.path.dirname(p)
        out.append(r)
    return out


def exp_results(exp, include_failed=False):
    res = sorted((r for r in all_results() if r["experiment"] == exp), key=lambda r: r["seed"])
    return res if include_failed else [r for r in res if not r["failed"]]


def has_weights(run):
    return os.path.exists(os.path.join(run["dir"], "best", "config.json"))


def rank_by_holdout(runs):
    return sorted(runs, key=lambda r: (-r["holdout_acc"], r["holdout_loss"]))


def chosen_run(exp):
    """Best successful run of an experiment by HOLD-OUT accuracy (decided without looking at dev)."""
    runs = [r for r in exp_results(exp) if has_weights(r)]
    assert runs, f"no successful runs with saved weights for {exp}"
    return rank_by_holdout(runs)[0]


# ---------------------------------------------------------------- outputs
def write_predictions(path, examples, logits):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    probs = torch.softmax(torch.tensor(logits), -1)[:, 0].tolist()
    preds = [int(np.argmax(l)) for l in logits]
    with open(path, "w") as f:
        for e, p in zip(examples, preds):
            f.write(json.dumps({"idx": e["idx"], "label": ID2LABEL[p]}) + "\n")
    with open(re.sub(r"\.jsonl$", "", path) + "_detailed.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["idx", "premise", "hypothesis", "gold", "pred", "p_entailment"])
        for e, p, pr in zip(examples, preds, probs):
            w.writerow([e["idx"], e["premise"], e["hypothesis"], ID2LABEL.get(e["label"], ""), ID2LABEL[p], round(pr, 4)])
    return preds


NEG = {"not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nor", "without", "cannot"}


def _words(t): return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", t.lower())


def slice_accuracy(examples, preds):
    def acc(mask):
        sel = [p == e["label"] for e, p, m in zip(examples, preds, mask) if m]
        return len(sel), (sum(sel) / len(sel) if sel else float("nan"))
    plen = [len(_words(e["premise"])) for e in examples]
    ovl = [len(set(_words(e["hypothesis"])) & set(_words(e["premise"]))) / max(len(set(_words(e["hypothesis"]))), 1)
           for e in examples]
    neg = [bool(NEG & set(_words(e["premise"] + " " + e["hypothesis"]))) for e in examples]
    return {"negation": acc(neg), "no negation": acc([not x for x in neg]),
            "premise <=30 words": acc([n <= 30 for n in plen]), "premise >30 words": acc([n > 30 for n in plen]),
            "overlap >=0.8": acc([o >= 0.8 for o in ovl]), "overlap <0.8": acc([o < 0.8 for o in ovl]),
            "gold entailment": acc([e["label"] == 0 for e in examples]),
            "gold not_entailment": acc([e["label"] == 1 for e in examples])}

In [ ]:
%%writefile {CODE_DIR}/experiments.py
"""Experiment grid. An experiment is named  <model>__<recipe>, e.g.  albert_large__freelb.
To try a new idea, add a recipe here (never change an existing one after its runs exist - results would mix)."""
import fnmatch
import glob
import json
import os

# Settings every experiment starts from (a model or a recipe can override any of them).
DEFAULTS = dict(
    max_len=256,            # tokens per (premise, hypothesis) pair
    epochs=10,
    evals_per_epoch=1,      # hold-out evaluations per epoch; the best one is the checkpoint we keep
    batch_size=16,
    grad_accum=1,
    lr=None,                # per model, see MODELS
    weight_decay=0.01,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    label_smoothing=0.0,
    val_ratio=0.1,          # 10% of train.jsonl held out to pick the checkpoint and to rank recipes (never dev)
    split_seed=1234,        # same hold-out split for every model, recipe and seed
    use_amp=True,           # bf16 autocast on GPUs that support it
    adv="none",             # "none" | "freelb" | "smart"
    adv_steps=2,            # FreeLB ascent steps per batch
    adv_init=0.01,          # initial perturbation, relative to each token's embedding norm
    adv_lr=0.03,            # ascent step, relative to each token's embedding norm
    adv_max=0.0,            # FreeLB: max perturbation (relative); 0 = unbounded
    adv_eps=0.02,           # SMART: perturbation radius (relative)
    smart_alpha=1.0,        # SMART: weight of the smoothness term
    teacher=None,           # distillation: name of a teacher file made by teacher.py
    kd_T=2.0,               # distillation temperature
    kd_alpha=0.7,           # weight of the teacher term on gold pairs (perturbed copies use the teacher only)
    model_kwargs={},
)

# Pretrained checkpoints only (self-supervised, no task head). mem_gb = rough GPU memory per run for sweep.py;
# small_gpu_micro = micro-batch used automatically on GPUs under 24 GB (T4, P100) - same effective batch of 16.
MODELS = {
    "albert_base":    dict(hub="albert/albert-base-v2",    family="albert", lr=2e-5, warmup_ratio=0.25, mem_gb=5),
    "albert_large":   dict(hub="albert/albert-large-v2",   family="albert", lr=1e-5, warmup_ratio=0.25, mem_gb=8),
    "albert_xlarge":  dict(hub="albert/albert-xlarge-v2",  family="albert", lr=1e-5, warmup_ratio=0.25, mem_gb=14, small_gpu_micro=8),
    "albert_xxlarge": dict(hub="albert/albert-xxlarge-v2", family="albert", lr=1e-5, warmup_ratio=0.25, mem_gb=28, small_gpu_micro=4),
    "deberta_xsmall": dict(hub="microsoft/deberta-v3-xsmall", family="deberta", lr=4e-5, mem_gb=5),
    "deberta_small":  dict(hub="microsoft/deberta-v3-small",  family="deberta", lr=3e-5, mem_gb=6),
    "deberta_base":   dict(hub="microsoft/deberta-v3-base",   family="deberta", lr=2e-5, mem_gb=12),
    "deberta_large":  dict(hub="microsoft/deberta-v3-large",  family="deberta", lr=6e-6, mem_gb=28, small_gpu_micro=4),
}

# Models made by pretrain.py (work/pretrained/<name>/) are added automatically, with the settings of the model they
# started from:  albert_large_rtd__freelb, albert_large_mlm__base, ...
_base = os.environ.get("RTE_WORK_DIR", "work")
for _meta in sorted(glob.glob(os.path.join(_base, "pretrained", "*", "pretrain_meta.json"))
                    + glob.glob(os.path.join(_base + "_smoke", "pretrained", "*", "pretrain_meta.json"))):
    _name = os.path.basename(os.path.dirname(_meta))
    _from = json.load(open(_meta))["base_model"]
    if _name not in MODELS and _from in MODELS:
        MODELS[_name] = {**MODELS[_from], "hub": "local:" + os.path.abspath(os.path.dirname(_meta)), "continued_from": _from}

FREELB = dict(adv="freelb", adv_steps=2, adv_init=0.01, adv_lr=0.03)
KD = dict(epochs=3, evals_per_epoch=4, warmup_ratio=0.1)    # 1 epoch = gold pairs + all perturbed copies (~9x data)

RECIPES = {
    "base":        {},                                              # our stable recipe (ALBERT: lr 1e-5, 25% warmup)
    "long":        dict(epochs=20, warmup_ratio=0.1),               # Mosbach et al. 2021: small lr, train longer
    "freelb":      dict(**FREELB),                                  # adversarial training on the embeddings
    "freelb_hi":   dict(adv="freelb", adv_steps=3, adv_init=0.02, adv_lr=0.1, adv_max=0.2),   # stronger perturbation
    "smart":       dict(adv="smart", adv_lr=0.02, adv_eps=0.02, smart_alpha=1.0),
    "kd":          dict(teacher="deberta_large", **KD),             # distil the DeBERTa-v3-large ensemble
    "kd_freelb":   dict(teacher="deberta_large", **KD, **FREELB),
    "self":        dict(teacher="self_{model}", **KD),              # born-again: teacher = ensemble of the same model
    "self_freelb": dict(teacher="self_{model}", **KD, **FREELB),
    "teacher":     dict(evals_per_epoch=2),                         # for the DeBERTa-v3-large teacher runs
}

ALL_EXPERIMENTS = [f"{m}__{r}" for m in MODELS for r in RECIPES]


def get_config(exp, smoke=False, micro_batch=None):
    if "__" not in exp:
        raise SystemExit(f"experiment names look like <model>__<recipe>, e.g. albert_large__freelb (got '{exp}')")
    model, recipe = exp.split("__", 1)
    if model not in MODELS:
        raise SystemExit(f"unknown model '{model}'. Models: {', '.join(MODELS)}")
    if recipe not in RECIPES:
        raise SystemExit(f"unknown recipe '{recipe}'. Recipes: {', '.join(RECIPES)}")
    m = MODELS[model]
    cfg = {**DEFAULTS, **{k: v for k, v in m.items() if k in DEFAULTS}, **RECIPES[recipe]}
    cfg.update(experiment=exp, model=model, recipe=recipe, hub=m["hub"], family=m["family"],
               mem_gb=m["mem_gb"] * (1.5 if cfg["adv"] == "smart" else 1.0))
    cfg["small_gpu_micro"] = m.get("small_gpu_micro")
    if cfg["teacher"]:
        cfg["teacher"] = cfg["teacher"].format(model=model)
    if micro_batch and micro_batch != cfg["batch_size"]:      # same effective batch, split into accumulated pieces
        eff = cfg["batch_size"] * cfg["grad_accum"]
        if eff % micro_batch:
            raise SystemExit(f"--micro-batch {micro_batch} must divide the effective batch {eff}")
        cfg["batch_size"], cfg["grad_accum"] = micro_batch, eff // micro_batch
    if smoke:
        cfg.update(epochs=1, evals_per_epoch=2)
    return cfg


def expand(patterns):
    """'albert_large__*' or '*__kd' -> matching experiment names (order kept, no duplicates)."""
    out = []
    for p in patterns:
        hits = [e for e in ALL_EXPERIMENTS if fnmatch.fnmatch(e, p)] if any(c in p for c in "*?[") else [p]
        if not hits:
            raise SystemExit(f"no experiment matches '{p}'")
        for h in hits:
            get_config(h)                      # validates the name
            if h not in out:
                out.append(h)
    return out

In [ ]:
%%writefile {CODE_DIR}/train.py
"""Train ONE run:  python train.py --exp albert_large__freelb --seed 42
Usually started by sweep.py; run it by hand to debug a single run (add --smoke for a 1-minute pipeline check)."""
import argparse

import common
from experiments import get_config

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True, help="<model>__<recipe>, see experiments.py")
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--smoke", action="store_true", help="tiny data + 1 epoch into work_smoke/ (checks the pipeline)")
ap.add_argument("--micro-batch", type=int, help="split each batch of 16 into pieces of this size (less GPU memory, same result)")
args = ap.parse_args()

if args.smoke:
    common.enable_smoke()
cfg = get_config(args.exp, smoke=args.smoke)
micro = args.micro_batch
if micro is None and common.gpu_memory_gb() < 24:            # 16 GB GPUs (T4, P100): smaller pieces, same batch of 16
    sizes = [m for m in (cfg["small_gpu_micro"], 8 if cfg["adv"] == "smart" else None) if m]   # SMART: 3 passes/step
    micro = min(sizes) if sizes else None
common.train_run(get_config(args.exp, smoke=args.smoke, micro_batch=micro), args.seed)

In [ ]:
%%writefile {CODE_DIR}/sweep.py
"""Run many experiments x seeds in parallel on one or more GPUs. Finished runs are skipped, so you can stop
(Ctrl-C) and restart any time.

    python sweep.py --exp albert_large__base albert_large__freelb --seeds 10
    python sweep.py --exp "albert_large__*" --seeds 20 --mem 120 --max-jobs 8
    python sweep.py --exp deberta_large__teacher --seeds 8 --gpus 0,1

Seeds are interleaved (seed 42 of every experiment first, then 43, ...), so if you stop early every
experiment has about the same number of seeds. Each run's log: work/runs/<experiment>/s<seed>/train.log"""
import argparse
import json
import os
import signal
import subprocess
import sys
import time
from collections import deque

import common
from experiments import expand, get_config

HERE = os.path.dirname(os.path.abspath(__file__))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", nargs="+", required=True, help="experiment names or patterns like 'albert_large__*'")
    ap.add_argument("--seeds", type=int, default=10, help="number of seeds per experiment")
    ap.add_argument("--first-seed", type=int, default=42)
    ap.add_argument("--gpus", default="0", help="comma-separated GPU ids, e.g. 0,1")
    ap.add_argument("--mem", type=float, default=120, help="GB of GPU memory to fill per GPU (H200: 141 GB)")
    ap.add_argument("--max-jobs", type=int, default=8, help="max parallel runs per GPU")
    ap.add_argument("--micro-batch", type=int, help="pass --micro-batch to every run (less memory, same result)")
    ap.add_argument("--smoke", action="store_true", help="tiny pipeline check into work_smoke/")
    args = ap.parse_args()
    if args.smoke:
        common.enable_smoke()

    exps = expand(args.exp)
    seeds = list(range(args.first_seed, args.first_seed + args.seeds))
    missing = sorted({get_config(e)["teacher"] for e in exps if get_config(e)["teacher"]
                      and not os.path.exists(common.teacher_path(get_config(e)["teacher"]))})
    if missing:
        skip = [e for e in exps if get_config(e)["teacher"] in missing]
        print(f"!! teacher file(s) missing: {missing} -> skipping {skip}\n   build them with teacher.py first")
        exps = [e for e in exps if e not in skip]
    jobs = deque((e, s) for s in seeds for e in exps
                 if not os.path.exists(os.path.join(common.run_dir(e, s), "results.json")))
    total = len(jobs)
    print(f"{len(exps)} experiment(s) x {len(seeds)} seed(s): {total} run(s) to do, "
          f"{len(exps) * len(seeds) - total} already finished")
    if not jobs:
        return

    gpus = [g.strip() for g in args.gpus.split(",") if g.strip()]
    used = {g: 0.0 for g in gpus}
    count = {g: 0 for g in gpus}
    running, retried, done, crashed = [], set(), 0, []

    def mem_of(job):
        return args.mem if job in retried else get_config(job[0])["mem_gb"]   # an OOM retry gets a GPU alone

    def start(job, gpu):
        exp, seed = job
        d = common.run_dir(exp, seed)
        os.makedirs(d, exist_ok=True)
        logf = open(os.path.join(d, "train.log"), "a")
        cmd = [sys.executable, os.path.join(HERE, "train.py"), "--exp", exp, "--seed", str(seed)] + (["--smoke"] if args.smoke else []) \
            + (["--micro-batch", str(args.micro_batch)] if args.micro_batch else [])
        env = {"OMP_NUM_THREADS": "4", **os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONUNBUFFERED": "1"}
        p = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env, cwd=os.getcwd())
        m = mem_of(job)
        used[gpu] += m
        count[gpu] += 1
        running.append({"p": p, "job": job, "gpu": gpu, "mem": m, "log": logf, "t0": time.time()})

    def stop_all(*_):
        print("\nstopping all runs ...")
        for r in running:
            r["p"].terminate()
        sys.exit(1)
    signal.signal(signal.SIGINT, stop_all)
    signal.signal(signal.SIGTERM, stop_all)

    deadline = float(os.environ.get("RTE_DEADLINE", "inf"))     # unix time; set by the Kaggle notebook (12 h sessions)
    while jobs or running:
        past_deadline = time.time() > deadline              # no new runs; the running ones finish
        if past_deadline and not running:
            print(f"\nsession time limit reached: {len(jobs)} run(s) left - run the same command in the next session")
            break
        # first-fit: launch every queued run that fits on some GPU (an oversized run starts on an idle GPU)
        for job in ([] if past_deadline else list(jobs)):
            for g in gpus:
                fits = used[g] + mem_of(job) <= args.mem and count[g] < args.max_jobs
                if fits or count[g] == 0:
                    jobs.remove(job)
                    start(job, g)
                    break
        time.sleep(3)
        for r in list(running):
            code = r["p"].poll()
            if code is None:
                continue
            running.remove(r)
            r["log"].close()
            used[r["gpu"]] -= r["mem"]
            count[r["gpu"]] -= 1
            exp, seed = r["job"]
            res_path = os.path.join(common.run_dir(exp, seed), "results.json")
            if code == 0 and os.path.exists(res_path):
                done += 1
                res = json.load(open(res_path))
                dev = f" | dev {res['dev_acc']:.4f}" if "dev_acc" in res else ""
                print(f"[{done}/{total}] {exp} s{seed} | hold-out {res['holdout_acc']:.4f}{dev} | epoch {res['best_epoch']}"
                      f" | {res['minutes']:.1f} min" + ("  FAILED (did not fit training data)" if res["failed"] else ""), flush=True)
                continue
            tail = open(os.path.join(common.run_dir(exp, seed), "train.log")).read()[-1500:]
            if "out of memory" in tail.lower() and r["job"] not in retried:
                retried.add(r["job"])
                jobs.append(r["job"])
                print(f"   {exp} s{seed}: out of GPU memory - will retry with a GPU to itself", flush=True)
                continue
            crashed.append(r["job"])
            print(f"!! {exp} s{seed} crashed (exit {code}). Last lines of its log:\n{tail}", flush=True)

    print(f"\nfinished: {done} run(s) trained, {len(crashed)} crashed {crashed if crashed else ''}")
    print("next: python summarize.py")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/teacher.py
"""Build a distillation teacher: the averaged predictions of the best finished runs of one or more experiments,
on the training pairs plus rule-based perturbed copies of them (no language model is used to make the copies).

    python teacher.py --name deberta_large --exp deberta_large__teacher --top 5
    python teacher.py --name self_albert_large --exp albert_large__freelb --top 5      # born-again (self) teacher

Students then use it through a recipe with teacher=<name> (see experiments.py: kd, kd_freelb, self, self_freelb).
Only train.jsonl is involved: the teacher runs were trained on the same 90% split, and the hold-out / dev numbers
printed here are for the report only."""
import argparse
import json
import os

import numpy as np
import torch

import common
from experiments import expand, get_config

ap = argparse.ArgumentParser()
ap.add_argument("--name", required=True, help="teacher name used by the recipes (deberta_large, self_albert_large, ...)")
ap.add_argument("--exp", nargs="+", required=True, help="experiment(s) whose runs form the teacher ensemble")
ap.add_argument("--top", type=int, default=5, help="use the best N successful runs by hold-out accuracy")
ap.add_argument("--n-aug", type=int, default=8, help="perturbed copies per training pair")
ap.add_argument("--aug-p", type=float, default=0.15, help="share of words deleted or masked in a copy")
ap.add_argument("--aug-seed", type=int, default=0)
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    common.enable_smoke()

exps = expand(args.exp)
runs = common.rank_by_holdout([r for e in exps for r in common.exp_results(e) if common.has_weights(r)])[: args.top]
if not runs:
    raise SystemExit(f"no successful finished runs for {exps} - train them first with sweep.py")
cfg = get_config(exps[0], smoke=args.smoke)
train_ex, hold_ex = common.load_split(cfg)
aug_ex = common.augment(train_ex, args.n_aug if not args.smoke else 1, args.aug_p, args.aug_seed)
all_ex = train_ex + aug_ex
dev_ex = common.load_dev()
print(f"teacher '{args.name}': {len(runs)} member(s) | {len(train_ex)} training pairs + {len(aug_ex)} perturbed copies")

probs = {"all": [], "hold": [], "dev": []}
for r in runs:
    model, tok = common.load_model_dir(os.path.join(r["dir"], "best"))
    rc = json.load(open(os.path.join(r["dir"], "best", "run_config.json")))
    dtype = common.amp_dtype(rc)
    for key, ex in (("all", all_ex), ("hold", hold_ex), ("dev", dev_ex)):
        if not ex:
            continue
        logits, _, _ = common.predict(model, common.loader(ex, tok, rc, batch_size=128), dtype)
        probs[key].append(torch.softmax(torch.tensor(logits), -1).numpy())
    print(f"  member {r['experiment']} s{r['seed']}: hold-out {r['holdout_acc']:.4f}"
          + (f", dev {r['dev_acc']:.4f}" if "dev_acc" in r else ""), flush=True)
    del model
    torch.cuda.empty_cache()

mean = {k: np.mean(v, axis=0) for k, v in probs.items() if v}          # ensemble = averaged probabilities
logp = np.log(np.clip(mean["all"], 1e-6, 1.0))
os.makedirs(common.teachers_dir(), exist_ok=True)
with open(common.teacher_path(args.name), "w") as f:
    for e, lp in zip(all_ex, logp):
        f.write(json.dumps({"idx": e["idx"], "premise": e["premise"], "hypothesis": e["hypothesis"],
                            "label": e["label"], "teacher_logp": [round(float(x), 6) for x in lp]}) + "\n")


def acc(p, ex): return float(np.mean([int(np.argmax(x)) == e["label"] for x, e in zip(p, ex)]))


gold_agree = acc(mean["all"][: len(train_ex)], train_ex)
aug_flip = float(np.mean([int(np.argmax(a)) != e["label"] for a, e in
                          zip(mean["all"][len(train_ex):], train_ex * max(1, len(aug_ex) // max(1, len(train_ex))))]))
meta = {"name": args.name, "members": [f"{r['experiment']}/s{r['seed']}" for r in runs], "n_train": len(train_ex),
        "n_aug": len(aug_ex), "aug_p": args.aug_p, "aug_seed": args.aug_seed,
        "ensemble_holdout_acc": acc(mean["hold"], hold_ex),
        "ensemble_dev_acc": acc(mean["dev"], dev_ex) if dev_ex else None,
        "agreement_with_gold_on_train": gold_agree,
        "share_of_copies_where_teacher_differs_from_original_label": aug_flip}
json.dump(meta, open(os.path.join(common.teachers_dir(), f"{args.name}_meta.json"), "w"), indent=2)
print(f"ensemble hold-out {meta['ensemble_holdout_acc']:.4f}"
      + (f" | ensemble dev {meta['ensemble_dev_acc']:.4f}" if dev_ex else "")
      + f" | agrees with gold on train {gold_agree:.3f} | differs from the original label on {100 * aug_flip:.1f}% of copies")
print(f"saved {common.teacher_path(args.name)}")

In [ ]:
%%writefile {CODE_DIR}/soup.py
"""Greedy model soup (Wortsman et al., 2022): average the WEIGHTS of several seeds of one experiment into a single
model with the same parameter count. Start from the best run by hold-out accuracy; add the next-best run only if
hold-out accuracy does not drop. Dev accuracy is measured once, at the end.

    python soup.py --exp albert_large__freelb albert_large__kd_freelb [--max 15]"""
import argparse
import json
import os

import torch

import common
from experiments import expand

ap = argparse.ArgumentParser()
ap.add_argument("--exp", nargs="+", required=True)
ap.add_argument("--max", type=int, default=15, help="consider at most the best N runs")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    common.enable_smoke()

dev_ex, test_ex = common.load_dev(), common.load_test()
for exp in expand(args.exp):
    runs = common.rank_by_holdout([r for r in common.exp_results(exp) if common.has_weights(r)])[: args.max]
    if len(runs) < 2:
        print(f"{exp}: soup skipped (fewer than 2 successful runs)")
        continue
    model, tok = common.load_model_dir(os.path.join(runs[0]["dir"], "best"))
    rc = json.load(open(os.path.join(runs[0]["dir"], "best", "run_config.json")))
    dtype = common.amp_dtype(rc)
    _, hold_ex = common.load_split(rc)
    hold_dl = common.loader(hold_ex, tok, rc)

    def state(run):
        return {k: v.float().cpu() for k, v in common.load_model_dir(os.path.join(run["dir"], "best"))[0].state_dict().items()}

    def holdout(sd, m=model):
        m.load_state_dict(sd)
        _, loss, acc = common.predict(m, hold_dl, dtype)
        return acc, loss

    soup, used = state(runs[0]), [runs[0]]
    best_acc, best_loss = holdout(soup)
    print(f"### {exp}: start s{runs[0]['seed']} hold-out {best_acc:.4f}")
    for r in runs[1:]:
        cand_sd, n = state(r), len(used)
        cand = {k: (soup[k] * n + cand_sd[k]) / (n + 1) if soup[k].is_floating_point() else soup[k] for k in soup}
        acc, loss = holdout(cand)
        keep = acc >= best_acc                     # Wortsman et al.: add if hold-out accuracy does not drop
        print(f"  + s{r['seed']:<5} hold-out {acc:.4f}  {'kept' if keep else 'rejected'}")
        if keep:
            soup, best_acc, best_loss = cand, acc, loss
            used.append(r)
    model.load_state_dict(soup)
    out = os.path.join(common.soups_dir(), exp)
    model.save_pretrained(out)
    tok.save_pretrained(out)
    json.dump({**rc, "soup_of": [r["seed"] for r in used]}, open(os.path.join(out, "run_config.json"), "w"), indent=2)
    res = {"experiment": exp, "kind": "soup", "ingredients": [r["seed"] for r in used], "holdout_acc": best_acc,
           "best_single_holdout_acc": runs[0]["holdout_acc"], "params": common.count_params(model),
           "model": rc["model"], "recipe": rc["recipe"], "family": rc["family"]}
    if dev_ex:
        res["dev_logits"], _, res["dev_acc"] = common.predict(model, common.loader(dev_ex, tok, rc), dtype)
    if test_ex:
        res["test_logits"], _, _ = common.predict(model, common.loader(test_ex, tok, rc), dtype)
    json.dump(res, open(os.path.join(out, "soup.json"), "w"))
    print(f"  soup of {len(used)} run(s): hold-out {best_acc:.4f}"
          + (f" | dev {res['dev_acc']:.4f} (best single run by hold-out: dev {runs[0].get('dev_acc', float('nan')):.4f})" if dev_ex else "")
          + f" -> {out}")
    del model
    torch.cuda.empty_cache()

In [ ]:
%%writefile {CODE_DIR}/shrink.py
"""Vocabulary pruning for DeBERTa-v3 models (most of their parameters are the 128K-token embedding table).
Keeps every token of train.jsonl + the K most common tokens + all single characters; any other word is split
into kept pieces. K is chosen on the HOLD-OUT split: the smallest K whose hold-out accuracy drops by at most
--tolerance. (ALBERT's embedding table is already small - 30K x 128 - so pruning it gains little.)

    python shrink.py --exp deberta_base__base [--use soup]"""
import argparse
import json
import os

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

import common
from experiments import expand


class PrunedVocab:
    """Encodes text for a model whose embedding table keeps only `keep_ids` (old ids, sorted). A token outside the
    kept set is split into the longest kept pieces of its text, e.g. '▁Hutchinson' -> '▁Hut' 'chin' 'son'."""

    def __init__(self, tok, keep_ids, max_len):
        self.tok, self.max_len = tok, max_len
        self.new_id = {old: new for new, old in enumerate(keep_ids)}
        special = set(tok.all_special_ids)
        self.str2new = {tok.convert_ids_to_tokens(old): new for old, new in self.new_id.items() if old not in special}
        self.max_piece = max(len(k) for k in self.str2new)
        self.unk, self.cls, self.sep = (self.new_id[tok.unk_token_id], self.new_id[tok.cls_token_id],
                                        self.new_id[tok.sep_token_id])
        assert self.new_id[tok.pad_token_id] == tok.pad_token_id, "pad token must keep its id"
        self.cache, self.stats = {}, {"tokens": 0, "split": 0, "unk": 0}

    def _decompose(self, piece):
        out, i = [], 0
        while i < len(piece):
            for j in range(min(len(piece), i + self.max_piece), i, -1):
                if piece[i:j] in self.str2new:
                    out.append(self.str2new[piece[i:j]])
                    i = j
                    break
            else:
                out.append(self.unk)
                i += 1
        return out

    def _map(self, ids):
        out = []
        for i in ids:
            self.stats["tokens"] += 1
            if i in self.new_id:
                out.append(self.new_id[i])
                continue
            if i not in self.cache:
                self.cache[i] = self._decompose(self.tok.convert_ids_to_tokens(i))
            self.stats["split"] += 1
            self.stats["unk"] += self.cache[i].count(self.unk)
            out.extend(self.cache[i])
        return out

    def encode(self, premise, hypothesis):
        p = self._map(self.tok(premise, add_special_tokens=False)["input_ids"])
        h = self._map(self.tok(hypothesis, add_special_tokens=False)["input_ids"])[: self.max_len - 3]
        p = p[: self.max_len - 3 - len(h)]
        ids = [self.cls] + p + [self.sep] + h + [self.sep]
        return {"input_ids": ids, "attention_mask": [1] * len(ids), "token_type_ids": [0] * (len(p) + 2) + [1] * (len(h) + 1)}


class PrunedDataset(Dataset):
    def __init__(self, examples, pv):
        self.items = [{**pv.encode(e["premise"], e["hypothesis"]),
                       "labels": e["label"] if e["label"] is not None else -100} for e in examples]
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]


def pruned_loader(examples, pv, tok):
    return DataLoader(PrunedDataset(examples, pv), batch_size=64, collate_fn=common.make_collate(tok))


def pruned_embedding(full_emb, keep_ids, pad_id):
    new = nn.Embedding(len(keep_ids), full_emb.embedding_dim, padding_idx=pad_id)
    new.weight.data.copy_(full_emb.weight.data[torch.tensor(keep_ids, device=full_emb.weight.device)])
    return new.to(full_emb.weight.device)


def keep_ids_for(tok, texts, top_k, V):
    ids = set(tok.all_special_ids) | set(range(min(top_k, V)))
    for p, h in texts:
        ids.update(tok(p, h)["input_ids"])
    n_tok = min(V, len(tok))
    for i, piece in enumerate(tok.convert_ids_to_tokens(list(range(n_tok)))):
        if piece is not None and len(piece.replace("▁", "")) <= 1:
            ids.add(i)
    return sorted(i for i in ids if i < V)


def load_pruned(path):
    model, tok = common.load_model_dir(path)
    meta = json.load(open(os.path.join(path, "vocab_keep.json")))
    return model, tok, PrunedVocab(tok, meta["keep_ids"], meta["max_len"])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", nargs="+", required=True)
    ap.add_argument("--use", choices=["best", "soup"], default="best", help="prune the best run (by hold-out) or the soup")
    ap.add_argument("--top-k", type=int, nargs="+", default=[5000, 10000, 20000, 40000])
    ap.add_argument("--tolerance", type=float, default=0.015, help="max hold-out accuracy drop (0.015 = ~4 examples)")
    ap.add_argument("--smoke", action="store_true")
    args = ap.parse_args()
    if args.smoke:
        common.enable_smoke()
    dev_ex, test_ex = common.load_dev(), common.load_test()

    for exp in expand(args.exp):
        src = os.path.join(common.soups_dir(), exp) if args.use == "soup" else os.path.join(common.chosen_run(exp)["dir"], "best")
        model, tok = common.load_model_dir(src)
        rc = json.load(open(os.path.join(src, "run_config.json")))
        if rc["family"] != "deberta":
            print(f"{exp}: skipped - pruning is meant for DeBERTa-v3 (128K vocabulary)")
            continue
        dtype = common.amp_dtype(rc)
        train_ex, hold_ex = common.load_split(rc)
        labelled = train_ex + hold_ex
        full_emb = model.get_input_embeddings()
        V = full_emb.num_embeddings
        full_params = common.count_params(model)
        _, _, ho_full = common.predict(model, common.loader(hold_ex, tok, rc), dtype)

        sweep = []
        for K in args.top_k:
            keep_tr = keep_ids_for(tok, [(e["premise"], e["hypothesis"]) for e in train_ex], K, V)   # for the choice
            model.set_input_embeddings(pruned_embedding(full_emb, keep_tr, tok.pad_token_id))
            _, _, ho = common.predict(model, pruned_loader(hold_ex, PrunedVocab(tok, keep_tr, rc["max_len"]), tok), dtype)
            keep = keep_ids_for(tok, [(e["premise"], e["hypothesis"]) for e in labelled], K, V)     # final vocabulary
            model.set_input_embeddings(pruned_embedding(full_emb, keep, tok.pad_token_id))
            row = {"top_k": K, "kept": len(keep), "params": common.count_params(model), "holdout_acc": ho, "_keep": keep}
            if dev_ex:
                pv = PrunedVocab(tok, keep, rc["max_len"])
                row["_dev_logits"], _, row["dev_acc"] = common.predict(model, pruned_loader(dev_ex, pv, tok), dtype)
                row["dev_tokens_split"] = pv.stats["split"] / max(pv.stats["tokens"], 1)
            sweep.append(row)
        ok = [r for r in sweep if r["holdout_acc"] >= ho_full - args.tolerance]
        best = min(ok, key=lambda r: r["params"]) if ok else max(sweep, key=lambda r: (r["holdout_acc"], -r["params"]))

        print(f"\n### {exp} ({args.use}): full {full_params / 1e6:.1f}M params, hold-out {ho_full:.4f}")
        for r in sweep:
            print(f"  top {r['top_k']:>6,} | vocab {r['kept']:>7,} | {r['params'] / 1e6:6.1f}M | hold-out {r['holdout_acc']:.4f}"
                  + (f" | dev {r['dev_acc']:.4f}" if dev_ex else "") + ("   <- chosen" if r is best else ""))

        model.set_input_embeddings(pruned_embedding(full_emb, best["_keep"], tok.pad_token_id))
        model.config.vocab_size = best["kept"]
        out = os.path.join(common.pruned_dir(), exp)
        model.save_pretrained(out)
        tok.save_pretrained(out)
        json.dump({**rc, "pruned_from": src}, open(os.path.join(out, "run_config.json"), "w"), indent=2)
        json.dump({"keep_ids": best["_keep"], "original_vocab_size": V, "top_k": best["top_k"], "max_len": rc["max_len"]},
                  open(os.path.join(out, "vocab_keep.json"), "w"))
        res = {"experiment": exp, "kind": "pruned", "source": args.use, "params": best["params"], "full_params": full_params,
               "top_k": best["top_k"], "vocab_kept": best["kept"], "holdout_acc_full": ho_full, "holdout_acc": best["holdout_acc"],
               "model": rc["model"], "recipe": rc["recipe"], "family": rc["family"],
               "sweep": [{k: v for k, v in r.items() if not k.startswith("_")} for r in sweep]}
        if dev_ex:
            res["dev_acc"], res["dev_logits"] = best["dev_acc"], best["_dev_logits"]
        if test_ex:
            res["test_logits"], _, _ = common.predict(model, pruned_loader(test_ex, PrunedVocab(tok, best["_keep"], rc["max_len"]), tok), dtype)
        json.dump(res, open(os.path.join(out, "pruned.json"), "w"))
        print(f"  saved {out}")
        del model, full_emb
        torch.cuda.empty_cache()

In [ ]:
%%writefile {CODE_DIR}/summarize.py
"""Everything trained so far, in one table + CSV files + an accuracy-vs-parameters plot.

    python summarize.py                       # all experiments
    python summarize.py --min-runs 5          # ladder uses only recipes with >= 5 successful runs

Recipes are RANKED BY HOLD-OUT mean (the 10% of train.jsonl), never by dev, so the dev numbers stay an honest
estimate. Outputs in work/summary/: experiments.csv, runs.csv, ladder.csv, accuracy_vs_params.png"""
import argparse
import csv
import glob
import json
import os
import statistics as st
import warnings

import common
from experiments import MODELS

ap = argparse.ArgumentParser()
ap.add_argument("--min-runs", type=int, default=3, help="successful runs a recipe needs to enter the ladder")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    common.enable_smoke()
warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    from scipy.stats import ttest_ind
except ImportError:
    ttest_ind = None

os.makedirs(common.summary_dir(), exist_ok=True)
runs = common.all_results()
if not runs:
    raise SystemExit("no finished runs yet")
order = {m: i for i, m in enumerate(MODELS)}
for _m, _spec in MODELS.items():                     # continued-pretraining models sort right after their base model
    if "continued_from" in _spec:
        order[_m] = order[_spec["continued_from"]] + 0.5


def mean_sd(xs):
    return (st.mean(xs), st.stdev(xs) if len(xs) > 1 else 0.0) if xs else (float("nan"), 0.0)


# ---------------------------------------------------------------- per experiment
by_exp = {}
for r in runs:
    by_exp.setdefault(r["experiment"], []).append(r)
rows = []
for exp, rs in by_exp.items():
    ok = [r for r in rs if not r["failed"]]
    dev = [r["dev_acc"] for r in ok if "dev_acc" in r]
    ho_m, ho_s = mean_sd([r["holdout_acc"] for r in ok])
    dv_m, dv_s = mean_sd(dev)
    chosen = common.rank_by_holdout(ok)[0] if ok else None
    rows.append({"experiment": exp, "family": rs[0]["family"], "model": rs[0]["model"], "recipe": rs[0]["recipe"],
                 "params": rs[0]["params"], "runs_ok": len(ok), "runs_total": len(rs),
                 "holdout_mean": ho_m, "holdout_sd": ho_s, "dev_mean": dv_m, "dev_sd": dv_s,
                 "dev_se": dv_s / len(dev) ** 0.5 if dev else float("nan"),
                 "chosen_seed": chosen["seed"] if chosen else None,
                 "chosen_dev": chosen.get("dev_acc") if chosen else None,
                 "minutes_per_run": st.mean(r["minutes"] for r in rs), "_dev": dev})
for row in rows:                                   # compare every recipe with the same model's "base" recipe
    ref = next((x for x in rows if x["experiment"] == f"{row['model']}__base"), None)
    row["dev_delta_vs_base"] = row["dev_mean"] - ref["dev_mean"] if ref and ref is not row else None
    row["p_vs_base"] = (float(ttest_ind(row["_dev"], ref["_dev"], equal_var=False).pvalue)
                        if ttest_ind and ref and ref is not row and len(row["_dev"]) > 1 and len(ref["_dev"]) > 1 else None)
rows.sort(key=lambda x: (x["family"], order.get(x["model"], 99), -x["holdout_mean"]))

# soups and pruned models (single models, so no spread)
extras = []
for kind, pattern in (("soup", os.path.join(common.soups_dir(), "*", "soup.json")),
                      ("pruned", os.path.join(common.pruned_dir(), "*", "pruned.json"))):
    for p in glob.glob(pattern):
        d = json.load(open(p))
        extras.append({"experiment": f"{d['experiment']} [{kind}]", "family": d["family"], "model": d["model"],
                       "recipe": d["recipe"], "params": d["params"], "holdout": d["holdout_acc"], "dev": d.get("dev_acc"),
                       "note": (f"{len(d['ingredients'])} runs averaged" if kind == "soup"
                                else f"vocab {d['vocab_kept']:,} (from {d['full_params'] / 1e6:.1f}M)")})


def pct(x): return "   -  " if x is None or x != x else f"{100 * x:6.2f}"


print(f"{'experiment':<36}{'params':>8}{'ok/runs':>9}{'hold-out mean+-sd':>19}{'dev mean+-sd':>17}{'+-SE':>7}"
      f"{'chosen':>8}{'dev vs base':>12}{'p':>7}{'min':>6}")
for r in rows:
    delta = "" if r["dev_delta_vs_base"] is None else f"{100 * r['dev_delta_vs_base']:+.2f}"
    p = "" if r["p_vs_base"] is None else f"{r['p_vs_base']:.3f}"
    print(f"{r['experiment']:<36}{r['params'] / 1e6:>7.1f}M{str(r['runs_ok']) + '/' + str(r['runs_total']):>9}"
          f"   {pct(r['holdout_mean'])} +-{pct(r['holdout_sd'])}  {pct(r['dev_mean'])} +-{pct(r['dev_sd'])}{pct(r['dev_se']):>7}"
          f"{pct(r['chosen_dev']):>8}{delta:>12}{p:>7}{r['minutes_per_run']:>6.1f}")
for e in extras:
    print(f"{e['experiment']:<36}{e['params'] / 1e6:>7.1f}M{'':>9}   {pct(e['holdout'])}{'':>10}{pct(e['dev'])}{'':>10}{e['note']}")
print("hold-out / dev = mean over successful runs; chosen = dev of the run with the best hold-out accuracy;"
      " p = Welch t-test of dev accuracies vs the same model's 'base' recipe")

with open(os.path.join(common.summary_dir(), "experiments.csv"), "w", newline="") as f:
    keys = [k for k in rows[0] if not k.startswith("_")]
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    w.writerows({k: r[k] for k in keys} for r in rows)
    for e in extras:
        w.writerow({"experiment": e["experiment"], "family": e["family"], "model": e["model"], "recipe": e["recipe"],
                    "params": e["params"], "holdout_mean": e["holdout"], "dev_mean": e["dev"]})
with open(os.path.join(common.summary_dir(), "runs.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["experiment", "seed", "failed", "best_epoch", "holdout_acc", "dev_acc", "train_ce_at_best", "minutes"])
    for r in sorted(runs, key=lambda r: (r["experiment"], r["seed"])):
        w.writerow([r["experiment"], r["seed"], r["failed"], r["best_epoch"], r["holdout_acc"], r.get("dev_acc"),
                    r["train_ce_at_best"], round(r["minutes"], 2)])

# ---------------------------------------------------------------- the ladder: best recipe per model size
ladder = []
for fam in sorted({r["family"] for r in rows}):
    for m in sorted({r["model"] for r in rows if r["family"] == fam}, key=lambda m: order.get(m, 99)):
        cands = [r for r in rows if r["model"] == m and r["runs_ok"] >= args.min_runs and r["recipe"] != "teacher"]
        cands = cands or [r for r in rows if r["model"] == m and r["runs_ok"] >= args.min_runs]
        if cands:
            ladder.append(max(cands, key=lambda r: r["holdout_mean"]))
print(f"\nLADDER - best recipe per model by HOLD-OUT mean (>= {args.min_runs} successful runs):")
print(f"{'family':<9}{'model':<24}{'recipe':<14}{'params':>9}{'hold-out':>10}{'dev mean+-sd':>17}{'chosen run dev':>16}")
prev = {}
for r in ladder:
    p = prev.get(r["family"])
    step = (f"   vs {p['model']}: x{r['params'] / p['params']:.1f} params, {100 * (r['dev_mean'] - p['dev_mean']):+.2f} dev"
            if p else "")
    print(f"{r['family']:<9}{r['model']:<24}{r['recipe']:<14}{r['params'] / 1e6:>8.1f}M{pct(r['holdout_mean']):>10}"
          f"{pct(r['dev_mean']):>10} +-{pct(r['dev_sd'])}{pct(r['chosen_dev']):>16}{step}")
    prev[r["family"]] = r
with open(os.path.join(common.summary_dir(), "ladder.csv"), "w", newline="") as f:
    keys = ["family", "model", "recipe", "params", "runs_ok", "holdout_mean", "dev_mean", "dev_sd", "dev_se", "chosen_seed", "chosen_dev"]
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    w.writerows({k: r[k] for k in keys} for r in ladder)

# ---------------------------------------------------------------- plot: dev accuracy vs parameters (one axis, log x)
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt and ladder:
    INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
    COLORS = {"albert": "#2a78d6", "deberta": "#eb6834"}          # validated categorical slots 1 and 2
    fig, ax = plt.subplots(figsize=(7, 4.2), dpi=200)
    labels = []
    for fam in sorted({r["family"] for r in ladder}):
        best_at = {}                       # one point per size: e.g. albert_large vs albert_large_rtd -> best by hold-out
        for r in ladder:
            if r["family"] == fam and r["dev_mean"] == r["dev_mean"]:
                k = round(r["params"] / 1e5)
                if k not in best_at or r["holdout_mean"] > best_at[k]["holdout_mean"]:
                    best_at[k] = r
        pts = sorted(best_at.values(), key=lambda r: r["params"])
        if not pts:
            continue
        xs = [r["params"] / 1e6 for r in pts]
        ys = [100 * r["dev_mean"] for r in pts]
        es = [100 * r["dev_se"] for r in pts]
        c = COLORS.get(fam, INK2)
        ax.plot(xs, ys, color=c, lw=2, zorder=2, label=f"{'ALBERT-v2' if fam == 'albert' else 'DeBERTa-v3'} (best recipe per size)")
        ax.errorbar(xs, ys, yerr=es, fmt="o", ms=8, color=c, mec="white", mew=2, elinewidth=1.5, capsize=3, zorder=3)
        labels.extend((x, y, f"{r['model'].split('_', 1)[1]} ({r['recipe']})") for x, y, r in zip(xs, ys, pts))
    ax.set_xscale("log")
    ticks = [t for t in (5, 10, 20, 50, 100, 200, 500, 1000)
             if min(r["params"] for r in ladder) / 2e6 <= t <= max(r["params"] for r in ladder) * 2 / 1e6]
    ax.set_xticks(ticks, [str(t) for t in ticks])
    ax.minorticks_off()
    ax.set_xlabel("parameters (millions, log scale)", color=INK2)
    ax.set_ylabel("dev accuracy, % (mean of seeds ± SE)", color=INK2)
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=8)
    ax.legend(frameon=False, fontsize=8, labelcolor=INK)
    # direct labels: try four positions around each point, keep the first that overlaps nothing already drawn
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    taken = [ax.get_legend().get_window_extent(rend)]
    taken += [ax.transData.transform_bbox(matplotlib.transforms.Bbox([[x * 0.97, y - 0.15], [x * 1.03, y + 0.15]]))
              for x, y, _ in labels]
    axbox = ax.get_window_extent(rend)
    for x, y, text in labels:
        for dx, dy, ha in ((7, -13, "left"), (-7, 7, "right"), (7, 7, "left"), (-7, -13, "right")):
            a = ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", ha=ha, fontsize=8, color=INK2)
            bb = a.get_window_extent(rend)
            inside = axbox.x0 <= bb.x0 and bb.x1 <= axbox.x1 and axbox.y0 <= bb.y0 and bb.y1 <= axbox.y1
            if inside and not any(bb.overlaps(t) for t in taken):
                break
            a.remove()
        else:
            a = ax.annotate(text, (x, y), xytext=(7, -13), textcoords="offset points", fontsize=8, color=INK2)
            bb = a.get_window_extent(rend)
        taken.append(bb)
    ax.set_title("RTE dev accuracy vs model size", color=INK, fontsize=11, loc="left")
    fig.tight_layout()
    out = os.path.join(common.summary_dir(), "accuracy_vs_params.png")
    fig.savefig(out)
    print(f"\nplot: {out}")
print(f"tables: {common.summary_dir()}/experiments.csv, runs.csv, ladder.csv")

In [ ]:
%%writefile {CODE_DIR}/predict.py
"""Predictions for a chosen model (one line per pair: {"idx": ..., "label": "entailment" | "not_entailment"}).

    python predict.py --exp albert_large__kd_freelb --split test                 # best run by hold-out
    python predict.py --exp albert_large__kd_freelb --use soup --split test      # the weight-averaged soup
    python predict.py --exp deberta_base__base --use pruned --split dev          # the vocabulary-pruned model
    python predict.py --exp albert_large__freelb --seed 45 --split dev           # one specific run

--split dev also prints accuracy, the confusion matrix and accuracy on slices (negation, length, word overlap)."""
import argparse
import json
import os

import common
from experiments import get_config

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True)
ap.add_argument("--use", choices=["best", "soup", "pruned"], default="best")
ap.add_argument("--seed", type=int, help="a specific run instead of the best one by hold-out")
ap.add_argument("--split", choices=["test", "dev"], default="test")
ap.add_argument("--out", help="output .jsonl (default: work/predictions/<exp>_<use>_<split>.jsonl)")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    common.enable_smoke()
get_config(args.exp)

examples = common.load_test() if args.split == "test" else common.load_dev()
if not examples:
    raise SystemExit(f"no {args.split} file in {common.DATA_DIR}/")
if args.use == "pruned":
    from shrink import load_pruned, pruned_loader
    path = os.path.join(common.pruned_dir(), args.exp)
    model, tok, pv = load_pruned(path)
    dl = pruned_loader(examples, pv, tok)
else:
    if args.use == "soup":
        path = os.path.join(common.soups_dir(), args.exp)
    elif args.seed is not None:
        path = os.path.join(common.run_dir(args.exp, args.seed), "best")
    else:
        path = os.path.join(common.chosen_run(args.exp)["dir"], "best")
    model, tok = common.load_model_dir(path)
rc = json.load(open(os.path.join(path, "run_config.json")))
if args.use != "pruned":
    dl = common.loader(examples, tok, rc)
logits, _, acc = common.predict(model, dl, common.amp_dtype(rc))

tag = f"{args.exp}_{args.use}" + (f"_s{args.seed}" if args.seed is not None else "")
out = args.out or os.path.join(common.preds_dir(), f"{tag}_{args.split}.jsonl")
preds = common.write_predictions(out, examples, logits)
print(f"{len(preds)} {args.split} predictions from {path} ({common.count_params(model):,} parameters) -> {out}")
if args.split == "dev":
    golds = [e["label"] for e in examples]
    cm = [[sum(1 for p, g in zip(preds, golds) if g == a and p == b) for b in (0, 1)] for a in (0, 1)]
    print(f"dev accuracy {acc:.4f} | confusion (rows gold, cols pred): entailment {cm[0]}  not_entailment {cm[1]}")
    for k, (n, a) in common.slice_accuracy(examples, preds).items():
        print(f"  {k:<22} n={n:<4} accuracy {a:.3f}")

In [ ]:
%%writefile {CODE_DIR}/download_models.py
"""Download the pretrained checkpoints once (before starting parallel runs) and check each against the rules:
the name must not suggest NLI/RTE/instruction tuning, and the checkpoint must NOT contain a classification head.
Also prints the parameter count of each model with its new 2-class head.

    python download_models.py                                   # every model in experiments.py
    python download_models.py albert_large albert_xxlarge deberta_large"""
import sys

from transformers import AutoModelForSequenceClassification, AutoTokenizer

import common
from experiments import MODELS

names = sys.argv[1:] or list(MODELS)
for key in names:
    hub = MODELS[key]["hub"]
    AutoTokenizer.from_pretrained(common.resolve_model(hub))
    model, info = AutoModelForSequenceClassification.from_pretrained(common.resolve_model(hub), num_labels=2,
                                                                     output_loading_info=True)
    head = common.check_rules(hub, info)
    print(f"{key:<16} {hub:<30} {common.count_params(model) / 1e6:7.1f}M params | rules OK, new head: {head}", flush=True)

In [ ]:
%%writefile {CODE_DIR}/clean_checkpoints.py
"""Free disk space (Kaggle keeps at most 20 GB of output): delete the saved weights (best/) of runs you no longer need.
results.json is always kept - hold-out, dev and test predictions of every run stay in summarize.py.

    python clean_checkpoints.py --keep 3                               # dry run: lists what would go
    python clean_checkpoints.py --keep 3 --yes                         # keep the 3 best runs (by hold-out) per experiment
    python clean_checkpoints.py --drop deberta_large__teacher --yes    # all weights of an experiment (after teacher.py)

Kept unless --drop names their experiment: the best --keep successful runs of each experiment and every member of a
teacher (teachers/*_meta.json). Failed runs lose their weights."""
import argparse
import glob
import json
import os
import shutil

import common
from experiments import expand

ap = argparse.ArgumentParser()
ap.add_argument("--keep", type=int, default=3, help="best runs to keep per experiment")
ap.add_argument("--drop", nargs="*", default=[], help="experiments whose weights are all deleted")
ap.add_argument("--yes", action="store_true", help="really delete (otherwise only show)")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
if args.smoke:
    common.enable_smoke()

drop = set(expand(args.drop)) if args.drop else set()
members = set()
for p in glob.glob(os.path.join(common.teachers_dir(), "*_meta.json")):
    members |= set(json.load(open(p))["members"])


def size(path):
    return sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(path) for f in fs)


by_exp = {}
for r in common.all_results():
    if common.has_weights(r):
        by_exp.setdefault(r["experiment"], []).append(r)
freed, n = 0, 0
for exp, runs in sorted(by_exp.items()):
    keep = set() if exp in drop else {r["seed"] for r in common.rank_by_holdout([r for r in runs if not r["failed"]])[: args.keep]}
    for r in runs:
        if r["seed"] in keep or (exp not in drop and f"{exp}/s{r['seed']}" in members):
            continue
        path = os.path.join(r["dir"], "best")
        b = size(path)
        freed += b
        n += 1
        print(f"{'deleting' if args.yes else 'would delete'} {path} ({b / 2**20:.0f} MB)")
        if args.yes:
            shutil.rmtree(path)
print(f"{'freed' if args.yes else 'would free'} {freed / 2**30:.2f} GB from {n} run(s)"
      + ("" if args.yes else " - add --yes to delete"))

## 2. Config - edit this cell before each session

In [ ]:
PHASES = ["check", "1"]   # this session's phases: "check", "1", "2", "3", "4", "5", "6"
SEEDS = 5                 # seeds per experiment (20 on the H200; 5 keeps a T4 week manageable)
N_AUG = 4                 # perturbed copies per training pair for distillation (8 on the H200)

# fill these in from the summary table (ranked by HOLD-OUT mean, not dev):
SELF_FROM = "freelb"      # phase 3: the 5 best runs of albert_large__<SELF_FROM> become the self-distillation teacher
R = "freelb"              # phase 4-6: best ALBERT-large recipe
R2 = "kd_freelb"          # phase 4: second best (also gets more seeds)
LADDER_SEEDS = 3          # phase 5: seeds for albert_base / albert_xlarge with recipe R
XXLARGE = False           # phase 5: ALBERT-xxlarge (223M) with the plain "base" recipe - slow on a T4 (~hours per run)
SUBMIT = ["albert_large__freelb", "albert_large__self_freelb", "albert_large__kd_freelb"]   # phase 6 (submitted: the first)
USE = "best"              # phase 6: "best" (best run by hold-out) or "soup" (weight average of the seeds)
SESSION_HOURS = 10.5      # start no new runs after this (Kaggle stops a session at 12 h)

## 3. Setup: data, earlier results, GPUs

In [ ]:
import glob, os, shutil, subprocess, sys, time
INPUT = os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input")
WORKING = os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working")
CODE_DIR, DATA, WORK = f"{WORKING}/rte_scripts", f"{WORKING}/data", f"{WORKING}/work"
os.environ.update(RTE_DATA_DIR=DATA, RTE_WORK_DIR=WORK, OMP_NUM_THREADS="2", TOKENIZERS_PARALLELISM="false",
                  RTE_DEADLINE=str(time.time() + SESSION_HOURS * 3600))

def sh(cmd):
    """Run a command and stream its output into the notebook."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")

def py(script, args=""):
    sh(f"cd {WORKING} && {sys.executable} {CODE_DIR}/{script} {args}")

# data files: anywhere in the inputs, except inside earlier outputs
os.makedirs(DATA, exist_ok=True)
for name, alts in (("train", ["train"]), ("val", ["val", "dev", "validation"]), ("test", ["test"])):
    hits = [p for a in alts for p in glob.glob(f"{INPUT}/**/{a}.jsonl", recursive=True)
            if not any(x in p for x in ("/work/", "/results/", "/predictions/", "/rte_scripts/"))]
    if hits:
        shutil.copy(sorted(hits, key=len)[0], f"{DATA}/{name}.jsonl")
        print(f"{name}: {sorted(hits, key=len)[0]}")
    elif name != "test":
        raise FileNotFoundError(f"{name}.jsonl not found under {INPUT} - add your dataset as an input")

# earlier sessions: copy their results back in (finished runs are skipped)
for runs in glob.glob(f"{INPUT}/**/work/runs", recursive=True):
    src = os.path.dirname(runs)
    for sub in ("runs", "teachers", "soups", "pruned"):
        if os.path.isdir(f"{src}/{sub}"):
            shutil.copytree(f"{src}/{sub}", f"{WORK}/{sub}", dirs_exist_ok=True)
    print(f"restored earlier results from {src}")
n_done = len(glob.glob(f"{WORK}/runs/*/s*/results.json"))
print(f"{n_done} finished run(s) available")

import torch
N_GPU = torch.cuda.device_count()
GPUS = ",".join(str(i) for i in range(N_GPU)) or "0"
print(f"GPUs: {[torch.cuda.get_device_name(i) for i in range(N_GPU)]} | torch {torch.__version__}")
SWEEP = f"--gpus {GPUS} --mem 15 --max-jobs 1"     # one run per T4 at a time

## Phase check: download models, check the rules, 2-minute pipeline test

In [ ]:
if "check" in PHASES:
    py("download_models.py", "albert_base albert_large albert_xlarge deberta_large" + (" albert_xxlarge" if XXLARGE else ""))
    py("sweep.py", f"--exp albert_large__freelb --seeds 2 --smoke {SWEEP}")

## Phase 1: ALBERT-large recipes + DeBERTa-v3-large teacher runs

In [ ]:
if "1" in PHASES:
    py("sweep.py", f"--exp albert_large__base albert_large__long albert_large__freelb albert_large__smart "
                   f"deberta_large__teacher --seeds {SEEDS} {SWEEP}")
    py("summarize.py")

## Phase 2: distillation from the DeBERTa-v3-large ensemble

In [ ]:
if "2" in PHASES:
    if not os.path.exists(f"{WORK}/teachers/deberta_large.jsonl"):
        py("teacher.py", f"--name deberta_large --exp deberta_large__teacher --top 5 --n-aug {N_AUG}")
    py("clean_checkpoints.py", "--drop deberta_large__teacher --yes")      # 1.7 GB each - the teacher file is all we need
    py("sweep.py", f"--exp albert_large__kd albert_large__kd_freelb --seeds {SEEDS} {SWEEP}")
    py("summarize.py")

## Phase 3: self-distillation (teacher = 5 best runs of `albert_large__<SELF_FROM>`)

In [ ]:
if "3" in PHASES:
    if not os.path.exists(f"{WORK}/teachers/self_albert_large.jsonl"):
        py("teacher.py", f"--name self_albert_large --exp albert_large__{SELF_FROM} --top 5 --n-aug {N_AUG}")
    py("sweep.py", f"--exp albert_large__self albert_large__self_freelb --seeds {SEEDS} {SWEEP}")
    py("summarize.py")

## Phase 4: confirm the top two recipes with twice the seeds

In [ ]:
if "4" in PHASES:
    py("sweep.py", f"--exp albert_large__{R} albert_large__{R2} --seeds {2 * SEEDS} {SWEEP}")
    py("summarize.py")

## Phase 5: size ladder (same architecture, different sizes)

In [ ]:
if "5" in PHASES:
    py("sweep.py", f"--exp albert_base__{R} albert_xlarge__{R} --seeds {LADDER_SEEDS} {SWEEP}")
    if XXLARGE:
        py("sweep.py", f"--exp albert_xxlarge__base --seeds 3 {SWEEP}")
    py("summarize.py")

## Phase 6: soups, final table, predictions for the 3 submissions

In [ ]:
if "6" in PHASES:
    py("soup.py", "--exp " + " ".join(SUBMIT))
    py("summarize.py")
    for exp in SUBMIT:
        py("predict.py", f"--exp {exp} --use {USE} --split dev")
        if os.path.exists(f"{DATA}/test.jsonl"):
            py("predict.py", f"--exp {exp} --use {USE} --split test")

## Always: free disk space (Kaggle keeps 20 GB of output) and collect the files to download

In [ ]:
py("clean_checkpoints.py", "--keep 3 --yes")
out = f"{WORKING}/results"
os.makedirs(out, exist_ok=True)
for p in glob.glob(f"{WORK}/summary/*") + glob.glob(f"{WORK}/teachers/*_meta.json") + glob.glob(f"{WORK}/predictions/*"):
    shutil.copy(p, out)
sh(f"du -sh {WORK} {out} 2>/dev/null; ls {out}")